# Credit Card Approval — Model Training & Comparison

This notebook mirrors the logic in `train.py`, but in an explorable, cell-by-cell format: preprocessing, class balancing, multi-model comparison, hyperparameter tuning, and evaluation.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

from utils.preprocessing import Preprocessor

df = pd.read_csv('../dataset/credit_card_approval.csv')
df.shape

## 1. Preprocess (clean + engineer + encode + scale)

In [ ]:
pre = Preprocessor()
X, y = pre.fit_transform(df)
X.shape, y.shape

## 2. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape

## 3. Balance classes with SMOTE (fallback to oversampling if unavailable)

In [ ]:
from train import balance_classes
X_train_bal, y_train_bal = balance_classes(X_train, y_train)
np.bincount(y_train_bal)

## 4. Compare candidate models

In [ ]:
from train import get_candidate_models, evaluate_model

models = get_candidate_models()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring='roc_auc', n_jobs=-1)
    model.fit(X_train_bal, y_train_bal)
    m = evaluate_model(model, X_test, y_test)
    rows.append({'model': name, 'cv_roc_auc': scores.mean(), **m})
    print(f"{name:<18} CV ROC-AUC={scores.mean():.4f}  Test ROC-AUC={m['roc_auc']:.4f}")

results = pd.DataFrame(rows).sort_values('roc_auc', ascending=False)
results[['model', 'cv_roc_auc', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]

## 5. Visualize the leaderboard

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(results['model'], results['roc_auc'], color='#5B7FF0')
plt.xlabel('Test ROC-AUC')
plt.title('Model Comparison')
plt.gca().invert_yaxis()
plt.show()

## 6. Hyperparameter tuning on the best model

In [ ]:
from train import tune_best_model

best_name = results.iloc[0]['model']
best_model = models[best_name]
tuned = tune_best_model(best_name, best_model, X_train_bal, y_train_bal)
tuned

## 7. Final evaluation

In [ ]:
y_pred = tuned.predict(X_test)
y_proba = tuned.predict_proba(X_test)[:, 1]

print('Test ROC-AUC:', roc_auc_score(y_test, y_proba))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Rejected', 'Approved']).plot(cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## 8. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr, color='#D9B36C', label=f'{best_name} (AUC={roc_auc_score(y_test, y_proba):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## Summary

This notebook reproduces the exact model-selection logic used by `train.py`. To persist a new model artifact for the Flask app, run `python train.py` from the project root rather than saving directly from this notebook, so `model/`, `scaler.pkl`, `encoder.pkl`, and `metrics.json` all stay in sync.